# Exploratory Job Market Analysis & Intelligence

**Project:** SAS Data Analytics Hackathon  
**Notebook:** `notebooks/02_job_market_analysis.ipynb`  
**Datasets:**  
1. `data/cleaned/DataScience_Jobs_cleaned.csv` (1,602 employer observations)  
2. `data/cleaned/Analytics_Jobs_cleaned.csv` (15,841 job postings)  

### Analytical Structure:
- **Part A:** Data Science Jobs Analysis (Titles, Hiring Volume, Salary Distribution, Experience Elasticity)
- **Part B:** Analytics Jobs Analysis (Designations, Locations, Salary Brackets, Skill Ecosystem, Salary Tiers)
- **Part C:** Combined Job Market Synthesis & Strategic Findings


In [ ]:
# Environment Setup & Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Visualization configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'figure.autolayout': True, 'figure.titlesize': 14, 'axes.titlesize': 12})

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df_ds = pd.read_csv(BASE_DIR / 'data' / 'cleaned' / 'DataScience_Jobs_cleaned.csv')
df_aj = pd.read_csv(BASE_DIR / 'data' / 'cleaned' / 'Analytics_Jobs_cleaned.csv')

print(f'Data Science Jobs shape: {df_ds.shape}')
print(f'Analytics Jobs shape: {df_aj.shape}')


## Part A: Data Science Jobs Analysis

### A.1 Most Common Job Titles & Openings Volume

In [ ]:
title_summary = df_ds.groupby('job_title').agg(
    record_count=('company_name', 'count'),
    total_openings=('num_of_jobs', 'sum'),
    mean_avg_salary=('avg_salary', 'mean'),
    median_min_exp=('min_experience', 'median')
).sort_values(by='total_openings', ascending=False)

title_summary


### A.2 Top Companies by Listed Jobs & Hiring Volume

In [ ]:
top_companies = df_ds.groupby('company_name').agg(
    roles_offered=('job_title', 'count'),
    total_openings=('num_of_jobs', 'sum'),
    mean_avg_salary=('avg_salary', 'mean')
).sort_values(by='total_openings', ascending=False).head(10)

top_companies


### A.3 Jobs with Highest Average Salary & Salary Distribution

In [ ]:
sal_by_role = df_ds.groupby('job_title')[['min_salary', 'avg_salary', 'max_salary']].mean().sort_values(by='avg_salary', ascending=False)
sal_by_role


In [ ]:
# Overall Salary Statistics (LPA)
df_ds[['min_salary', 'avg_salary', 'max_salary']].describe().round(2)


### A.4 Correlation Analysis: Experience vs Salary & Volume vs Salary

In [ ]:
exp_sal_corr = df_ds['min_experience'].corr(df_ds['avg_salary'])
jobs_sal_corr = df_ds['num_of_jobs'].corr(df_ds['avg_salary'])

print(f'Pearson correlation (min_experience vs avg_salary): {exp_sal_corr:.3f}')
print(f'Pearson correlation (num_of_jobs vs avg_salary): {jobs_sal_corr:.3f}')


### A.5 Experience Requirements by Job Title

In [ ]:
df_ds.groupby('job_title')['min_experience'].describe().sort_values(by='50%', ascending=False)


## Part B: Analytics Jobs Analysis

### B.1 Top Job Designations & Top Locations

In [ ]:
print('Top 10 Job Designations:')
print(df_aj['job_desig'].value_counts().head(10))

print('\nTop 10 Locations:')
print(df_aj['location'].value_counts().head(10))


### B.2 Salary Distribution & Experience Cross-Tabulation

In [ ]:
sal_exp_summary = df_aj.groupby('salary').agg(
    postings_count=('s_no', 'count'),
    share_percentage=('s_no', lambda x: round(len(x)/len(df_aj)*100, 2)),
    mean_min_exp=('min_experience', 'mean'),
    mean_max_exp=('max_experience', 'mean'),
    mean_avg_exp=('avg_experience', 'mean'),
    salary_order=('salary_bracket_order', 'first')
).sort_values(by='salary_order')

sal_exp_summary


### B.3 Frequently Appearing Skills from `key_skills`

In [ ]:
all_skills = []
for s_str in df_aj['key_skills'].dropna():
    for token in str(s_str).split(','):
        ct = token.strip().lower()
        if ct and not ct.startswith('.') and (len(ct) > 1 or ct == 'r') and ct != 'not specified':
            all_skills.append(ct)

top_skills_series = pd.Series(Counter(all_skills)).sort_values(ascending=False).head(20)
top_skills_df = pd.DataFrame({'frequency': top_skills_series, 'penetration_pct': (top_skills_series / len(df_aj) * 100).round(2)})
top_skills_df


### B.4 Skills Associated with Higher Salary Ranges vs Entry Ranges

In [ ]:
skill_sal_data = []
for _, row in df_aj.iterrows():
    s_str = row['key_skills']
    sal_mid = row['avg_salary']
    sal_brk = row['salary']
    if pd.isnull(s_str) or s_str == 'Not Specified':
        continue
    for t in str(s_str).split(','):
        ct = t.strip().lower()
        if ct and not ct.startswith('.') and (len(ct) > 1 or ct == 'r'):
            skill_sal_data.append({'skill': ct, 'avg_salary': sal_mid, 'salary': sal_brk})

df_skill_sal = pd.DataFrame(skill_sal_data)
skill_tier_analysis = df_skill_sal.groupby('skill').agg(
    mentions=('avg_salary', 'count'),
    mean_salary=('avg_salary', 'mean'),
    high_bracket_pct=('salary', lambda x: round((x.isin(['15to25', '25to50'])).mean() * 100, 2)),
    low_bracket_pct=('salary', lambda x: round((x.isin(['0to3', '3to6'])).mean() * 100, 2))
).query('mentions >= 100').sort_values(by='mean_salary', ascending=False)

skill_tier_analysis.head(15)


## Part C: Visualizations Gallery

All figures generated and saved to `outputs/figures/job_market/`.

In [ ]:
# Display saved figures summary
fig_dir = BASE_DIR / 'outputs' / 'figures' / 'job_market'
fig_files = sorted(list(fig_dir.glob('*.png')))
for f in fig_files:
    print(f'Figure generated: {f.name} ({f.stat().st_size / 1024:.1f} KB)')
